<a href="https://colab.research.google.com/github/Developer3009/Codesoft-Internship/blob/SPAM-SMS-DETECTION/SPAM_SMS_DETECTION.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer

# 1. Load the dataset (handling the specific encoding of this file)
df = pd.read_csv('spam.csv', encoding='latin-1')

# 2. Clean up the dataframe
# Keep only the first two columns and rename them for clarity
df = df.iloc[:, [0, 1]]
df.columns = ['label', 'text']

# 3. Encode the labels (ham = 0, spam = 1)
df['label'] = df['label'].map({'ham': 0, 'spam': 1})

print("Dataset shape:", df.shape)
print("\nClass distribution:\n", df['label'].value_counts(normalize=True) * 100)

# 4. Separate Features (X) and Target (y)
X = df['text']
y = df['label']

# 5. Stratified Train-Test Split (Ensures 86/14 ham/spam ratio is maintained)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

print(f"\nTraining messages: {len(X_train)} | Testing messages: {len(X_test)}")

In [ ]:
from sklearn.naive_bayes import MultinomialNB
from sklearn.svm import SVC

# 1. Initialize TF-IDF Vectorizer
# We remove common English stop words ("and", "the", "is") as they don't help identify spam
tfidf = TfidfVectorizer(stop_words='english', max_features=5000)

# Fit on training data and transform both sets
X_train_tfidf = tfidf.fit_transform(X_train)
X_test_tfidf = tfidf.transform(X_test)

# 2. Initialize the Models
models = {
    "Naive Bayes": MultinomialNB(),
    # A linear kernel is highly recommended for text data because text features are linearly separable in high dimensions
    "Support Vector Machine": SVC(kernel='linear', random_state=42)
}

predictions = {}

# 3. Train and predict
for name, model in models.items():
    print(f"Training {name}...")
    model.fit(X_train_tfidf, y_train)
    predictions[name] = model.predict(X_test_tfidf)
    print(f"{name} trained successfully.\n")

In [ ]:
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt

for name, y_pred in predictions.items():
    print(f"=== {name} Performance ===")
    print(f"Accuracy: {accuracy_score(y_test, y_pred):.4f}\n")

    # Classification report
    print(classification_report(y_test, y_pred, target_names=['Ham (0)', 'Spam (1)']))

    # Confusion Matrix
    cm = confusion_matrix(y_test, y_pred)
    plt.figure(figsize=(5,3))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Greens', cbar=False)
    plt.title(f'{name} - Confusion Matrix')
    plt.xlabel('Predicted Label')
    plt.ylabel('True Label')
    plt.show()
    print("="*50, "\n")